In [1]:
# ==============================================================================
# CELL 0: OPUS BOOTSTRAP (SETUP, CONNECTIVITY & AESTHETICS)
# ==============================================================================
# Purpose: 1. Silence warnings for a clean output.
#          2. Mount Google Drive and connect to the Golden Master DB.
#          3. Apply the "Opus Lab" Visual Canon (Standardized Aesthetics).
# ==============================================================================

# --- 1. HYGIENE PROTOCOL ---
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
import pandas as pd
pd.options.mode.chained_assignment = None  # Silence SettingWithCopy

# --- 2. IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import sqlite3
from google.colab import drive

# --- 3. CONNECTIVITY ---
print("⏳ Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")
except:
    print("ℹ️ Drive already mounted or running locally.")

# DEFINITIVE PATH
DB_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'

if not os.path.exists(DB_PATH):
    print(f"🔴 CRITICAL: Database not found at {DB_PATH}")
else:
    print(f"✅ Database found: {DB_PATH}")
    db_engine = create_engine(f'sqlite:///{DB_PATH}')
    print("✅ SQL Engine Active.")

# --- 4. VISUAL CANON (OPUS LAB THEME) ---
OPUS_PURPLE = '#440154'
OPUS_TEAL   = '#21918c'
OPUS_GREY   = '#FAFAFA'
OPUS_TEXT   = '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY,
    'axes.facecolor': OPUS_GREY,
    'text.color': OPUS_TEXT,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.edgecolor': '#DDDDDD',
    'grid.color': '#E0E0E0',
    'font.family': 'sans-serif',
    'axes.titlecolor': OPUS_PURPLE,
    'axes.titleweight': 'bold',
    'figure.titlesize': 24,
    'figure.titleweight': 'bold'
})

print("✅ Visual Identity Loaded: Opus Lab (Light Mode).")
print("\n--- SYSTEM READY ---")

⏳ Mounting Google Drive...
Mounted at /content/drive
✅ Drive Mounted.
✅ Database found: /content/drive/MyDrive/_Pienza/Assets/Database/opus.db
✅ SQL Engine Active.
✅ Visual Identity Loaded: Opus Lab (Light Mode).

--- SYSTEM READY ---


In [2]:
# ==============================================================================
# CELL 1: CLOUD COMMAND CENTER (AUTH, IDS & BQ CONFIGURATION)
# ==============================================================================
# Purpose: 1. Authenticate with Google Cloud.
#          2. Define Project and Dataset IDs.
#          3. Initialize the BigQuery Client.
#          4. Ensure the target Dataset exists in the Cloud.
# ==============================================================================

from google.colab import auth
from google.cloud import bigquery

# --- 1. AUTHENTICATION ---
print("🔐 Solicitando acceso a Google Cloud...")
auth.authenticate_user()
print("✅ Autenticación exitosa.")

# --- 2. CONFIGURATION (VERIFICA TUS IDS) ---
PROJECT_ID = 'drivers-dilemma' # <--- Asegúrate que este sea tu Project ID actual
DATASET_ID = 'pienza_mini'    # El contenedor de tu Star Schema

# --- 3. INITIALIZE CLIENT ---
client = bigquery.Client(project=PROJECT_ID)
print(f"🚀 Cliente BigQuery activo en proyecto: {PROJECT_ID}")

# --- 4. DATASET READINESS ---
def prepare_dataset():
    dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
    try:
        client.get_dataset(dataset_ref)
        print(f"✅ El dataset '{DATASET_ID}' ya existe. Listo para recibir datos.")
    except Exception:
        print(f"🏗️ Dataset '{DATASET_ID}' no encontrado. Creándolo...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US" # O tu región preferida
        client.create_dataset(dataset, timeout=30)
        print(f"✅ Dataset '{DATASET_ID}' creado exitosamente.")

prepare_dataset()
print("\n--- INFRAESTRUCTURA LISTA ---")

🔐 Solicitando acceso a Google Cloud...
✅ Autenticación exitosa.
🚀 Cliente BigQuery activo en proyecto: drivers-dilemma
🏗️ Dataset 'pienza_mini' no encontrado. Creándolo...
✅ Dataset 'pienza_mini' creado exitosamente.

--- INFRAESTRUCTURA LISTA ---


In [3]:
# ==============================================================================
# CELL 16: THE FINAL MIGRATION (De Local a Soberanía Cloud)
# ==============================================================================
import sqlite3
import pandas as pd
from google.cloud import bigquery

def complete_migration():
    print(f"🚀 Iniciando migración de {DB_PATH} a BigQuery Nativo...")

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]

    client = bigquery.Client(project=PROJECT_ID)
    # Configuración para sobrescribir si ya existen y detectar esquemas automáticamente
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

    for table in tables:
        print(f"   ⬆️ Migrando: {table}...")
        df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

        # Limpieza de nombres de columna (BQ es estricto)
        df.columns = [c.lower().replace(' ', '_').replace('-', '_').replace('.', '_') for c in df.columns]

        table_id = f"{PROJECT_ID}.{DATASET_ID}.{table}"

        # Carga directa desde DataFrame
        job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
        job.result() # Esperar confirmación
        print(f"   ✅ {table} ya es NATIVA en BigQuery.")

    conn.close()
    print("\n🏆 Misión Cumplida. El Star Schema vive en la infraestructura de Google.")

complete_migration()

🚀 Iniciando migración de /content/drive/MyDrive/_Pienza/Assets/Database/opus.db a BigQuery Nativo...
   ⬆️ Migrando: driver_state_at_request...
   ✅ driver_state_at_request ya es NATIVA en BigQuery.
   ⬆️ Migrando: heuristic_flag...
   ✅ heuristic_flag ya es NATIVA en BigQuery.
   ⬆️ Migrando: interpolation_quality...
   ✅ interpolation_quality ya es NATIVA en BigQuery.
   ⬆️ Migrando: offer_action...
   ✅ offer_action ya es NATIVA en BigQuery.
   ⬆️ Migrando: outcome...
   ✅ outcome ya es NATIVA en BigQuery.
   ⬆️ Migrando: post_offer_status...
   ✅ post_offer_status ya es NATIVA en BigQuery.
   ⬆️ Migrando: product_category...
   ✅ product_category ya es NATIVA en BigQuery.
   ⬆️ Migrando: reason_primary...
   ✅ reason_primary ya es NATIVA en BigQuery.
   ⬆️ Migrando: record_status...
   ✅ record_status ya es NATIVA en BigQuery.
   ⬆️ Migrando: raw_offers_ocr...
   ✅ raw_offers_ocr ya es NATIVA en BigQuery.
   ⬆️ Migrando: heuristic_flag_offers...
   ✅ heuristic_flag_offers ya es NAT

In [4]:
# ==============================================================================
# CELL 17: OPUS CLOUD AUDIT (RELATIONAL INTEGRITY CHECK)
# ==============================================================================
# Propósito: Verificar que las relaciones del ERD sobrevivieron a la migración.
#            Busca registros huérfanos y valida la conectividad del Star Schema.
# ==============================================================================

def execute_relational_audit():
    print(f"🔍 Iniciando Auditoría Forense en Dataset: {DATASET_ID}...\n")

    # Lista de pruebas: (Nombre de la prueba, Query SQL)
    audits = [
        ("1. TOTAL OFFERS",
         f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_ID}.offers`"),

        ("2. INTEGRIDAD DE PRODUCTOS (Orphans)",
         f"""SELECT COUNT(o.offer_id) as huerfanos
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` p
             ON o.product_category_fk = p.product_category_id
             WHERE p.product_category_id IS NULL"""),

        ("3. CONECTIVIDAD OFFERS <-> FEATURES",
         f"""SELECT COUNT(o.offer_id) as total_links
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             INNER JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef
             ON o.offer_id = ef.offer_id_fk"""),

        ("4. INTEGRIDAD DE ACCIONES (Orphans)",
         f"""SELECT COUNT(o.offer_id) as huerfanos
             FROM `{PROJECT_ID}.{DATASET_ID}.offers` o
             LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.offer_action` oa
             ON o.offer_action_fk = oa.offer_action_id
             WHERE oa.offer_action_id IS NULL"""),

        ("5. PUENTE HEURISTIC FLAGS (Linkage)",
         f"""SELECT COUNT(*) as total_flags
             FROM `{PROJECT_ID}.{DATASET_ID}.heuristic_flag_offers` hfo
             JOIN `{PROJECT_ID}.{DATASET_ID}.heuristic_flag` hf
             ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id""")
    ]

    for title, query in audits:
        try:
            res = client.query(query).to_dataframe()
            val = res.iloc[0, 0]

            # Lógica de semáforo
            status = "✅ OK"
            if "Orphans" in title and val > 0:
                status = f"❌ ERROR: {val} huérfanos encontrados!"
            if "TOTAL OFFERS" in title:
                status = f"📊 {val} registros"
            if "total_links" in title and val < 4000: # Ajusta según tu N real
                status = f"⚠️ ALERTA: Solo {val} conexiones encontradas."

            print(f"{title:<40} {status}")
        except Exception as e:
            print(f"❌ Error en prueba '{title}': {e}")

    print("\n🏁 AUDITORÍA FINALIZADA.")

execute_relational_audit()

🔍 Iniciando Auditoría Forense en Dataset: pienza_mini...

1. TOTAL OFFERS                          📊 4765 registros
2. INTEGRIDAD DE PRODUCTOS (Orphans)     ✅ OK
3. CONECTIVIDAD OFFERS <-> FEATURES      ✅ OK
4. INTEGRIDAD DE ACCIONES (Orphans)      ✅ OK
5. PUENTE HEURISTIC FLAGS (Linkage)      ✅ OK

🏁 AUDITORÍA FINALIZADA.


In [5]:
# ==============================================================================
# CELL 18: VIEW LOGIC EXTRACTION (The Source Recipes)
# ==============================================================================
import sqlite3

def extract_view_logic():
    print(f"📖 Extrayendo definiciones de vistas desde {DB_PATH}...\n")

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Buscamos específicamente las Vistas
    cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='view';")
    views = cursor.fetchall()

    if not views:
        print("⚠️ No se encontraron vistas en la base de datos.")
    else:
        for name, sql in views:
            print(f"--- VISTA: {name} ---")
            print(sql)
            print("\n" + "="*50 + "\n")

    conn.close()

extract_view_logic()

📖 Extrayendo definiciones de vistas desde /content/drive/MyDrive/_Pienza/Assets/Database/opus.db...

--- VISTA: v_trip_funnel_metrics ---
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - j

In [6]:
# ==============================================================================
# CELL 21: VIEW FORGER - v_trip_funnel_metrics
# ==============================================================================
# Purpose: Calculates precise durations between T0-T4 lifecycle events.
# Logic:   Pivots long-format 'trip_events' into a wide performance record.
# ==============================================================================

view_name = "v_trip_funnel_metrics"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        -- Capturamos los 5 pilares del tiempo con SAFE_CAST para integridad
        MAX(CASE WHEN event_types_id_fk = 1 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t4_completed,
        -- Capturamos los pilares financieros
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(realized_fare AS FLOAT64) END) AS realized_fare
    FROM
        `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,

    -- TRADUCCIÓN: julianday delta -> TIMESTAMP_DIFF (BigQuery Standard)
    TIMESTAMP_DIFF(p.t2_arrived, p.t1_accepted, SECOND) AS duration_to_pickup_sec,
    TIMESTAMP_DIFF(p.t3_started, p.t2_arrived, SECOND) AS duration_waiting_sec,
    TIMESTAMP_DIFF(p.t4_completed, p.t3_started, SECOND) AS duration_trip_sec,
    TIMESTAMP_DIFF(p.t4_completed, p.t1_accepted, SECOND) AS duration_total_engagement_sec,

    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_trip_funnel_metrics...
✅ ÉXITO: v_trip_funnel_metrics ahora es nativa en BigQuery.


In [7]:
# ==============================================================================
# CELL 22: VIEW FORGER - v_trip_funnel_wide
# ==============================================================================
# Purpose: Collapses the long-format event log into a wide format (one row/trip).
# Logic:   Identifies T0-T4 timestamps and critical fares per trip ID.
# ==============================================================================

view_name = "v_trip_funnel_wide"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    trip_id_legacy,
    MAX(offer_id_fk) AS offer_id_fk,
    -- Captura de Timestamps con SAFE_CAST para asegurar formato de nube
    MAX(CASE WHEN event_types_id_fk = 1 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) AS t4_timestamp,
    -- Captura de Fares
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(realized_fare AS FLOAT64) END) AS realized_fare
FROM `{PROJECT_ID}.{DATASET_ID}.trip_events`
GROUP BY trip_id_legacy
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_trip_funnel_wide...
✅ ÉXITO: v_trip_funnel_wide ahora es nativa en BigQuery.


In [8]:
# ==============================================================================
# CELL 23: VIEW FORGER - v_trip_final_kpis
# ==============================================================================
# Purpose: Calculates core financial KPIs (Spread %, EPH on ride, EPH total).
# Logic:   Converts T1-T4 intervals into hourly rates and profit indices.
# ==============================================================================

view_name = "v_trip_final_kpis"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,

    -- DURACIONES: Reemplazo de julianday por TIMESTAMP_DIFF
    TIMESTAMP_DIFF(v.t2_timestamp, v.t1_timestamp, SECOND) AS duration_to_pickup_sec,
    TIMESTAMP_DIFF(v.t3_timestamp, v.t2_timestamp, SECOND) AS duration_waiting_sec,
    TIMESTAMP_DIFF(v.t4_timestamp, v.t3_timestamp, SECOND) AS duration_trip_sec,
    TIMESTAMP_DIFF(v.t4_timestamp, v.t1_timestamp, SECOND) AS total_engagement_duration_sec,

    v.upfront_fare,
    v.realized_fare,

    -- SPREAD: Usamos SAFE_DIVIDE por elegancia y seguridad
    SAFE_DIVIDE(v.realized_fare, v.upfront_fare) AS spread_percentage,

    -- EPH ON RIDE: Realized Fare / (Trip Duration in Hours)
    SAFE_DIVIDE(
        v.realized_fare,
        (TIMESTAMP_DIFF(v.t4_timestamp, v.t3_timestamp, SECOND) / 3600.0)
    ) AS eph_on_ride,

    -- EPH TOTAL TIME: Realized Fare / (Total Engagement in Hours)
    SAFE_DIVIDE(
        v.realized_fare,
        (TIMESTAMP_DIFF(v.t4_timestamp, v.t1_timestamp, SECOND) / 3600.0)
    ) AS eph_total_time

FROM `{PROJECT_ID}.{DATASET_ID}.v_trip_funnel_wide` v
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_trip_final_kpis...
✅ ÉXITO: v_trip_final_kpis ahora es nativa en BigQuery.


In [9]:
# ==============================================================================
# CELL 24: VIEW FORGER - v_mission_dossier
# ==============================================================================
# Purpose: Forges the final "Mission File" linking Offer IDs to realized KPIs.
# Logic:   Aggregates offer links from events and merges with financial results.
# ==============================================================================

view_name = "v_mission_dossier"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH OfferLink AS (
    -- Aislar el vínculo único entre el viaje y su oferta origen
    SELECT
        trip_id_legacy,
        MAX(offer_id_fk) AS offer_id
    FROM
        `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY
        trip_id_legacy
)
SELECT
    ol.offer_id,
    kpi.*
FROM
    `{PROJECT_ID}.{DATASET_ID}.v_trip_final_kpis` AS kpi
LEFT JOIN
    OfferLink AS ol ON kpi.trip_id_legacy = ol.trip_id_legacy
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_mission_dossier...
✅ ÉXITO: v_mission_dossier ahora es nativa en BigQuery.


In [10]:
# ==============================================================================
# CELL 25: VIEW FORGER - v_broche_fks
# ==============================================================================
# Purpose: Full data lineage audit. Connects the 6 stages of the data lifecycle.
# Logic:   Left joins all major tables using Offer and Lifetime IDs as anchors.
# ==============================================================================

view_name = "v_broche_fks"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    -- 1. THE ORIGIN (OCR): Los datos crudos de la imagen
    r.ocr_id                  AS raw_ocr_id,

    -- 2. THE HUB (Offers): La tabla maestra de decisiones
    o.offer_id                AS hub_offer_id,
    o.session_fk              AS hub_session_fk,
    o.ocr_fk                  AS hub_ocr_fk,

    -- 3. THE BRAIN (Features): La capa de inteligencia procesada
    ef.feature_id             AS feat_id,
    ef.offer_id_fk            AS feat_offer_id_fk,

    -- 4. THE FIELD (Trip Events): La recolección de alta fidelidad (T0-T4)
    te.event_id               AS event_id,
    te.offer_id_fk            AS event_offer_id_fk,

    -- 5. THE CORPORATE RECORD (Lifetime Trips): El registro oficial de Uber
    lt.lifetime_trips_id      AS lt_id,
    lt.offer_id_fk            AS lt_offer_id_fk,

    -- 6. THE BANK (Earnings): El recibo final de pago neto
    ae.activity_earnings_id   AS ae_id,
    ae.offer_id_fk            AS ae_offer_id_fk,
    ae.lifetime_trips_fk      AS ae_lt_fk

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.raw_offers_ocr` r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.trip_events` te          ON o.offer_id = te.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_broche_fks...
✅ ÉXITO: v_broche_fks ahora es nativa en BigQuery.


In [11]:
# ==============================================================================
# CELL 26: VIEW FORGER - v_offers_human
# ==============================================================================
# Purpose: The "Universal Translator". Decodes IDs into human-readable strings.
# Logic:   Joins the main Offers table with all 8 Dimension tables.
# ==============================================================================

view_name = "v_offers_human"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    -- 1. IDENTITY & TIME
    o.offer_id,
    o.session_fk,
    o.offer_timestamp,

    -- 2. CORE METRICS (The Physics)
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,

    -- 3. HUMAN READABLE LABELS (The Translation)
    pc.category_name                  AS str_product,
    oa.offer_action_description       AS str_action,
    rp.reason_primary_description     AS str_reason,
    ds.driver_state_at_request_description AS str_driver_state,
    pos.post_offer_status_description AS str_post_status,
    out_dim.outcome_description       AS str_outcome,
    iq.interpolation_quality_description AS str_interp_quality,
    rs.record_status_description      AS str_record_status,

    -- 4. GEOSPATIAL CONTEXT
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat, o.pickup_lon,
    o.dropoff_lat, o.dropoff_lon,

    -- 5. FLAGS & NOTES
    o.is_surge, o.surge_amount,
    o.is_turbo_plus, o.turbo_plus_amount,
    o.is_reservation, o.reservation_amount,
    o.special_note_raw,

    -- 6. THE BRAIN (Engineered Features - ERD ALIGNED)
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_60sec,
    ef.cycle_avg_dtp_km,
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,
    ef.eph_direct,
    ef.eph_operational,
    ef.is_operational_downgrade,

    -- ML Track Predictions
    ef.eph_realized_ML,
    ef.eph_complete_ML,
    ef.is_spread_downgrade_ML,
    ef.is_total_cycle_downgrade_ML,

    -- Strategic & Temporal Context
    ef.home_vector_alignment_score,
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    -- Las 8 Dimensiones del Significado (Aseguramos nombres de tablas correctos)
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` pc        ON o.product_category_fk = pc.product_category_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.offer_action` oa            ON o.offer_action_fk = oa.offer_action_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.reason_primary` rp          ON o.reason_primary_fk = rp.reason_primary_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.driver_state_at_request` ds ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.post_offer_status` pos      ON o.post_offer_status_fk = pos.post_offer_status_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.outcome` out_dim            ON o.outcome_fk = out_dim.outcome_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.interpolation_quality` iq   ON o.interpolation_quality_fk = iq.interpolation_quality_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.record_status` rs           ON o.record_status_fk = rs.record_status_id

    -- La Capa de Inteligencia (Features)
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef     ON o.offer_id = ef.offer_id_fk
"""

try:
    print(f"🏗️ Forjando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora es nativa en BigQuery.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista: v_offers_human...
✅ ÉXITO: v_offers_human ahora es nativa en BigQuery.


In [12]:
# ==============================================================================
# CELL 31: FORENSIC REPAIR - v_lifecycle_audit (FULL PARITY EDITION)
# ==============================================================================
# Propósito: Restaurar las 11 columnas faltantes para alcanzar las 35 del ERD.
#            Incluye comparaciones de texto (OCR vs Clean) y timestamps crudos.
# ==============================================================================

view_name = "v_lifecycle_audit"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
WITH TripEventsPivot AS (
    SELECT
        offer_id_fk,
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 2 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN SAFE_CAST(event_timestamp AS TIMESTAMP) END) as t4_timestamp,
        MAX(upfront_fare) as te_upfront_fare,
        MAX(SAFE_CAST(realized_fare AS FLOAT64)) as te_realized_fare
    FROM `{PROJECT_ID}.{DATASET_ID}.trip_events`
    GROUP BY offer_id_fk, trip_id_legacy
),
HistoryStats AS (
    SELECT
        lt.offer_id_fk,
        lt.lifetime_trips_id,
        lt.original_fare,
        ae.net_earning,
        SUM(lt.original_fare) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as cum_uber_earnings,
        SUM(ae.net_earning) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as cum_net_earnings,
        AVG(SAFE_DIVIDE(ae.net_earning, lt.original_fare)) OVER (ORDER BY SAFE_CAST(lt.request_timestamp AS TIMESTAMP) ROWS UNBOUNDED PRECEDING) as rolling_avg_net_take_rate
    FROM `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae ON lt.lifetime_trips_id = ae.lifetime_trips_fk
)
SELECT
    -- 1. IDENTIFIERS (3 cols)
    o.offer_id, o.session_fk, te.trip_id_legacy,

    -- 2. TIME CONSISTENCY (4 cols)
    r.time_taken                AS ocr_raw_time,
    SAFE_CAST(o.offer_timestamp AS TIMESTAMP) AS clean_timestamp,
    ef.day_of_week, ef.time_of_day_block,

    -- 3. PRODUCT CONSISTENCY (4 cols)
    r.ride_type                 AS ocr_product,
    pc.category_name            AS internal_product,
    ae.product_category         AS bank_product,
    lt.global_product_name      AS official_product,

    -- 4. SPATIAL CONSISTENCY - PICKUPS (3 cols)
    r.pickup_address            AS ocr_pickup,
    o.pickup_address            AS clean_pickup,
    ef.pickup_ambiguity,

    -- 5. SPATIAL CONSISTENCY - DROPOFFS (3 cols)
    r.dropoff_address           AS ocr_dropoff,
    o.dropoff_address           AS clean_dropoff,
    ef.dropoff_ambiguity,

    -- 6. FINANCIAL CONSISTENCY (6 cols)
    lt.original_fare            AS uber_original_fare,
    r.upfront_fare              AS ocr_upfront,
    o.upfront_fare              AS clean_upfront,
    te.te_upfront_fare          AS events_upfront,
    te.te_realized_fare         AS events_realized,
    ae.net_earning              AS bank_net_earning,

    -- 7. TEMPORAL AUDIT - BASE TIMESTAMPS & DELTAS (9 cols)
    te.t1_timestamp             AS gts_t1_accepted,
    SAFE_CAST(lt.request_timestamp AS TIMESTAMP) AS uber_request,
    TIMESTAMP_DIFF(te.t1_timestamp, SAFE_CAST(lt.request_timestamp AS TIMESTAMP), SECOND) AS delta_accept_sec,

    te.t3_timestamp             AS gts_t3_started,
    SAFE_CAST(lt.pickup_timestamp AS TIMESTAMP) AS uber_pickup,
    TIMESTAMP_DIFF(te.t3_timestamp, SAFE_CAST(lt.pickup_timestamp AS TIMESTAMP), SECOND) AS delta_start_sec,

    te.t4_timestamp             AS gts_t4_completed,
    SAFE_CAST(lt.dropoff_timestamp AS TIMESTAMP) AS uber_dropoff,
    TIMESTAMP_DIFF(te.t4_timestamp, SAFE_CAST(lt.dropoff_timestamp AS TIMESTAMP), SECOND) AS delta_end_sec,

    -- 8. HISTORICAL CONTEXT (3 cols)
    hist.cum_uber_earnings, hist.cum_net_earnings, hist.rolling_avg_net_take_rate

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.raw_offers_ocr` r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.product_category` pc     ON o.product_category_fk = pc.product_category_id
    LEFT JOIN TripEventsPivot te      ON o.offer_id = te.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.lifetime_trips` lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.activity_earnings` ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk
    LEFT JOIN HistoryStats hist       ON o.offer_id = hist.offer_id_fk
"""

try:
    print(f"🏗️ Reparando vista: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} ahora tiene las 35 columnas reglamentarias.")

    # También actualizamos la vista aceptada (que depende de esta)
    accepted_view_id = f"{PROJECT_ID}.{DATASET_ID}.v_lifecycle_audit_accepted"
    accepted_sql = f"CREATE OR REPLACE VIEW `{accepted_view_id}` AS SELECT * FROM `{view_id}` WHERE trip_id_legacy IS NOT NULL"
    client.query(accepted_sql).result()
    print(f"✅ ÉXITO: v_lifecycle_audit_accepted actualizada automáticamente.")

except Exception as e:
    print(f"❌ FALLO en la reparación: {e}")

🏗️ Reparando vista: v_lifecycle_audit...
✅ ÉXITO: v_lifecycle_audit ahora tiene las 35 columnas reglamentarias.
✅ ÉXITO: v_lifecycle_audit_accepted actualizada automáticamente.


In [13]:
# ==============================================================================
# CELL 28: VIEW FORGER - v_lifecycle_audit_accepted
# ==============================================================================
# Purpose: Filters the master audit to show ONLY successfully completed rides.
# Logic:   Identifies rows with valid Trip IDs and orders them chronologically.
# ==============================================================================

view_name = "v_lifecycle_audit_accepted"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    *
FROM
    `{PROJECT_ID}.{DATASET_ID}.v_lifecycle_audit`
WHERE
    trip_id_legacy IS NOT NULL -- Filtro exclusivo para misiones completadas
ORDER BY
    clean_timestamp DESC
"""

try:
    print(f"🏗️ Forjando la vista final: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO: {view_name} activada.")
    print("\n🏁 MIGRACIÓN LÓGICA COMPLETADA AL 100%.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando la vista final: v_lifecycle_audit_accepted...
✅ ÉXITO: v_lifecycle_audit_accepted activada.

🏁 MIGRACIÓN LÓGICA COMPLETADA AL 100%.


In [14]:
# ==============================================================================
# CELL 29: VIEW FORGER - v_ML_Supervised (Absolute Parity Edition)
# ==============================================================================
# Purpose: The definitive flat table for Machine Learning operations.
# Logic:   Joins Offers, Features, Silver Palette, and Heuristic Context.
# No summarization. No omissions. 100% Structural Parity.
# ==============================================================================

view_name = "v_ML_Supervised"
view_id = f"{PROJECT_ID}.{DATASET_ID}.{view_name}"

sql_logic = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    -- 1. BASE: Table 'offers' (o)
    o.offer_id,
    o.session_fk,
    o.ocr_fk,
    o.image_content_hash,
    o.offer_timestamp,
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat,
    o.pickup_lon,
    o.dropoff_lat,
    o.dropoff_lon,
    o.is_surge,
    o.surge_amount,
    o.is_turbo_plus,
    o.turbo_plus_amount,
    o.is_reservation,
    o.reservation_amount,
    o.is_priority,
    o.priority_amount,
    o.is_exclusive,
    o.is_vip,
    o.is_identity_verified,
    o.is_long_trip,
    o.is_multiple_destinations,
    o.is_teens,
    o.rider_star_rating,
    o.rider_trip_count,
    o.time_in_session_sec,
    o.session_progress_ratio,
    o.inferred_agent_lat,
    o.inferred_agent_lon,
    o.inferred_agent_bearing,
    o.inferred_agent_speed_mps,
    o.is_imputed,
    o.special_note_raw,
    o.comment_1,
    o.comment_2,
    o.product_category_fk,
    o.offer_action_fk,
    o.reason_primary_fk,
    o.post_offer_status_fk,
    o.driver_state_at_request_fk,
    o.outcome_fk,
    o.interpolation_quality_fk,
    o.record_status_fk,

    -- 2. INTELLIGENCE: Table 'engineered_features' (ef)
    ef.feature_id,
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_10sec,
    ef.offer_density_30sec,
    ef.offer_density_60sec,
    ef.offer_density_180sec,
    ef.consecutive_rejects,
    ef.cycle_avg_dtp_km,
    ef.cycle_std_dtp_km,
    ef.cycle_ttp_dtp_ratio,
    ef.dispatch_lead_time_sec,
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,
    ef.eph_direct,
    ef.eph_direct_index,
    ef.eph_direct_label,
    ef.eph_operational,
    ef.eph_operational_index,
    ef.eph_operational_label,
    ef.is_operational_downgrade,
    ef.eph_realized_ML,
    ef.eph_realized_index_ML,
    ef.eph_realized_label_ML,
    ef.is_spread_downgrade_ML,
    ef.eph_complete_ML,
    ef.eph_complete_index_ML,
    ef.eph_complete_label_ML,
    ef.is_total_cycle_downgrade_ML,
    ef.eph_realized_EDA,
    ef.eph_realized_index_EDA,
    ef.eph_realized_label_EDA,
    ef.is_spread_downgrade_EDA,
    ef.eph_complete_EDA,
    ef.eph_complete_index_EDA,
    ef.eph_complete_label_EDA,
    ef.is_total_cycle_downgrade_EDA,
    ef.home_vector_alignment_score,
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type,

    -- 3. GEOSPATIAL: Table 'silver_palette' (sp)
    sp.dropoff_polygon_id,
    sp.dropoff_polygon_name,
    sp.dropoff_h3_hex_id,
    sp.dropoff_hdbscan_id,
    sp.dropoff_hdbscan_name,
    sp.realized_traffic_index,
    sp.historical_rolling_avg_traffic_index,
    sp.traffic_volatility_index_ml,
    sp.traffic_volatility_index_eda,

    -- 4. CONTEXT: Table 'heuristic_flag' (hf) via 'heuristic_flag_offers' (hfo)
    hf.heuristic_flag_description AS heuristic_flag_context

FROM
    `{PROJECT_ID}.{DATASET_ID}.offers` o
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.engineered_features` ef ON o.offer_id = ef.offer_id_fk
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.silver_palette` sp ON o.offer_id = sp.offer_id
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.heuristic_flag_offers` hfo ON o.offer_id = hfo.offers_offer_id
LEFT JOIN
    `{PROJECT_ID}.{DATASET_ID}.heuristic_flag` hf ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id
"""

try:
    print(f"🏗️ Forjando vista integral: {view_name}...")
    client.query(sql_logic).result()
    print(f"✅ ÉXITO TOTAL: {view_name} está reconstruida con el 100% de sus columnas.")
except Exception as e:
    print(f"❌ FALLO CRÍTICO en {view_name}: {e}")

🏗️ Forjando vista integral: v_ML_Supervised...
✅ ÉXITO TOTAL: v_ML_Supervised está reconstruida con el 100% de sus columnas.


In [15]:
# ==============================================================================
# CELL 30: THE GRAND PARITY AUDIT (SQLite vs. BigQuery)
# ==============================================================================
# Propósito: Certificar paridad 1:1 en Filas y Columnas para Tablas y Vistas.
# Logic:     Interrogates metadata from both systems and flags discrepancies.
# ==============================================================================

def perform_parity_audit():
    print(f"⚖️ Iniciando Auditoría de Paridad Estructural...")
    print(f"   Fuente: {DB_PATH}")
    print(f"   Destino: {PROJECT_ID}.{DATASET_ID}\n")

    # --- 1. INVENTARIO SQLITE (Local) ---
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Obtenemos tablas y vistas
    cursor.execute("SELECT name, type FROM sqlite_master WHERE name NOT LIKE 'sqlite_%';")
    local_objects = cursor.fetchall()

    sqlite_results = []
    for name, obj_type in local_objects:
        # Conteo de Columnas
        cursor.execute(f"PRAGMA table_info('{name}')")
        col_count = len(cursor.fetchall())
        # Conteo de Filas
        cursor.execute(f"SELECT COUNT(*) FROM '{name}'")
        row_count = cursor.fetchone()[0]

        sqlite_results.append({
            'name': name.lower(),
            'type': obj_type.upper(),
            'sq_rows': row_count,
            'sq_cols': col_count
        })
    conn.close()
    df_sqlite = pd.DataFrame(sqlite_results)

    # --- 2. INVENTARIO BIGQUERY (Cloud) ---
    # Usamos INFORMATION_SCHEMA para obtener metadata de BQ
    bq_query = f"""
        SELECT
            table_name as name,
            table_type as type,
            -- BigQuery no guarda el row_count de vistas en metadata, hay que consultarlo
        FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
    """
    df_bq_meta = client.query(bq_query).to_dataframe()

    bq_results = []
    for _, row in df_bq_meta.iterrows():
        name = row['name']
        obj_type = "TABLE" if row['type'] == "BASE TABLE" else "VIEW"

        # Conteo de Columnas via API
        table_ref = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{name}")
        col_count = len(table_ref.schema)

        # Conteo de Filas via Query (necesario para vistas)
        row_query = f"SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.{name}`"
        row_count = client.query(row_query).to_dataframe().iloc[0,0]

        bq_results.append({
            'name': name.lower(),
            'bq_rows': row_count,
            'bq_cols': col_count
        })
    df_bq = pd.DataFrame(bq_results)

    # --- 3. CONSOLIDACIÓN Y COMPARACIÓN ---
    audit = pd.merge(df_sqlite, df_bq, on='name', how='outer')

    # Lógica de Validación
    audit['row_match'] = audit['sq_rows'] == audit['bq_rows']
    audit['col_match'] = audit['sq_cols'] == audit['bq_cols']

    def get_status(row):
        if pd.isna(row['bq_rows']): return "❌ FALTANTE EN BQ"
        if row['row_match'] and row['col_match']: return "✅ MATCH"
        res = ""
        if not row['row_match']: res += f"Δ Filas ({row['bq_rows'] - row['sq_rows']}) "
        if not row['col_match']: res += f"Δ Cols ({row['bq_cols'] - row['sq_cols']}) "
        return "⚠️ MISMATCH: " + res

    audit['status'] = audit.apply(get_status, axis=1)

    # --- 4. REPORTE VISUAL ---
    print(audit[['name', 'type', 'sq_rows', 'bq_rows', 'sq_cols', 'bq_cols', 'status']].to_string(index=False))

    mismatches = audit[audit['status'] != "✅ MATCH"].shape[0]
    if mismatches == 0:
        print("\n🏆 CERTIFICACIÓN DE PARIDAD: EXITOSA. El Star Schema es 1:1.")
    else:
        print(f"\n🛑 ATENCIÓN: Se detectaron {mismatches} discrepancias. Revisar log.")

perform_parity_audit()

⚖️ Iniciando Auditoría de Paridad Estructural...
   Fuente: /content/drive/MyDrive/_Pienza/Assets/Database/opus.db
   Destino: drivers-dilemma.pienza_mini

                      name  type  sq_rows  bq_rows  sq_cols  bq_cols  status
         activity_earnings TABLE     3446     3446        7        7 ✅ MATCH
   driver_state_at_request TABLE        2        2        2        2 ✅ MATCH
       engineered_features TABLE     4765     4765       45       45 ✅ MATCH
               event_types TABLE        5        5        3        3 ✅ MATCH
            heuristic_flag TABLE        8        8        2        2 ✅ MATCH
     heuristic_flag_offers TABLE      831      831        2        2 ✅ MATCH
     interpolation_quality TABLE        5        5        2        2 ✅ MATCH
            lifetime_trips TABLE     3446     3446       15       15 ✅ MATCH
              offer_action TABLE        2        2        2        2 ✅ MATCH
                    offers TABLE     4765     4765       50       50 ✅ MAT